In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import sys

sys.path.insert(0, r"E:\Drive\NOA\MBD-Prediction\Modules")

from utils import describe_dataframe, process_greek

enc = 'utf-8'

pd.set_option('display.max_columns', None)

In [2]:
NUTS2 = 'Thessaly'
NUTS2_ID = 'EL61'

In [3]:
base_folder = 'E:/Drive/NOA/MBD-Prediction/Grid Level/data/'

In [4]:
for year_item in range (2009,2011):
    data = pd.read_csv(f'{base_folder}/{NUTS2}/GR_{NUTS2}_Timeline_GRID_{year_item}_filled_mean_rain.csv', encoding=enc)
    greece_nuts3 = gpd.read_file(r"E:\Drive\NOA\MBD-Prediction\Europe NUTS - LAU Shapefiles\Greece\data\EL_Whole.shp",  encoding=enc)
    greece_lau1 = gpd.read_file(r"E:\Drive\NOA\MBD-Prediction\Shapefiles\Greece Shapefiles\el_lau1.shp",  encoding=enc)
    print(f'Year: {year_item}')
    try:
        data.drop(columns=['Unnamed: 0'], inplace=True)
    except Exception:
        print("No column named \'Unnamed: 0\'")

    try:
        data.drop(columns=['rainfall.1'], inplace=True)
    except Exception:
        print("No column named \'rainfall.1\'")

    greece_grid = gpd.GeoDataFrame(data, geometry=gpd.points_from_xy(data.x, data.y), crs='EPSG:4326')

    greece_nuts3 = greece_nuts3.to_crs(epsg=4326)
    greece_lau1 = greece_lau1.to_crs(epsg=4326)
    greece_grid = greece_grid.to_crs(epsg=4326)

    greece_lau1.drop(columns=['KWD_YPES'], inplace = True)
    greece_lau1.drop(greece_lau1[greece_lau1['NAME'] == 'Άθως'].index, inplace = True)
    greece_lau1.rename(columns={"NAME": "LAU1_NAME", "geometry": "lau1_geom"}, inplace = True)
    greece_lau1.set_geometry("lau1_geom", inplace = True)
    greece_lau1['lau1_centroid'] = greece_lau1['lau1_geom'].centroid
    greece_lau1['lau1_rpoint'] = greece_lau1['lau1_geom'].representative_point()
    greece_lau1.reset_index(drop=True, inplace=True)

    greece_nuts3.rename(columns={"geometry": "nuts3_geom"}, inplace = True)
    greece_nuts3.set_geometry("nuts3_geom", inplace = True)    

    print(f'greece_nuts3 CRS: {greece_nuts3.crs}')
    print(f'greece_lau1 CRS: {greece_lau1.crs}')
    print(f'greece_grid CRS: {greece_grid.crs}')

    greece_lau1.set_geometry("lau1_rpoint", inplace = True)
    greece_nuts3 = greece_nuts3.to_crs(epsg=4326)
    greece_lau1 = greece_lau1.to_crs(epsg=4326)

    greece_gdf = greece_lau1.sjoin(greece_nuts3, predicate="within", how = "left")
    greece_gdf.drop(columns = ['lau1_centroid',	'lau1_rpoint', 'index_right', 'CNTR_CODE', 'CNTR_NAME', 'NUTS1_LATN', 'NUTS2_LATN', 'NUTS3_LATN'], inplace = True)
    greece_gdf.set_geometry('lau1_geom', inplace = True)
    greece_grid.set_geometry('geometry', inplace = True)
    greece_gdf = greece_gdf.to_crs(epsg=4326)
    greece_grid = greece_grid.to_crs(epsg=4326)

    greece_gdf_sub = greece_gdf[greece_gdf['NUTS2_ID'] == NUTS2_ID].copy()
    greece_gdf_sub.reset_index(drop=True, inplace=True)

    cm_grid = greece_grid.sjoin_nearest(greece_gdf, how='left', max_distance=0.01, distance_col='dist')
    cm_grid.rename(columns={'NUTS2_ID_left': 'NUTS2_ID'}, inplace=True)

    new_order = ['x', 'y', 'cell_number', 'dt_placement','NUTS1_ID', 'NUTS1_NAME', 'NUTS2_ID', 'NUTS2_NAME', 'NUTS3_ID', 'NUTS3_NAME', 'LAU1_NAME', 'year', 'month',
             'ndvi', 'ndmi', 'ndwi', 'ndbi', 'ndvi_mean', 'ndmi_mean',
             'ndwi_mean', 'ndbi_mean', 'ndvi_std', 'ndmi_std', 'ndwi_std',
             'ndbi_std', 'lst_day', 'lst_night', 'rainfall', 'day',
             'lst_day_jan_mean', 'lst_night_jan_mean', 'lst_day_feb_mean',
             'lst_night_feb_mean', 'lst_day_mar_mean', 'lst_night_mar_mean',
             'lst_day_apr_mean', 'lst_night_apr_mean', 'acc_rainfall_1week',
             'acc_rainfall_2week', 'acc_rainfall_month', 'acc_rainfall_jan',
             'MOUNT_TYPE','URBN_TYPE', 'COAST_TYPE']
    
    cm_grid = cm_grid[new_order]

    process_greek(cm_grid, 'NUTS1_NAME')
    process_greek(cm_grid, 'NUTS2_NAME')
    process_greek(cm_grid, 'NUTS3_NAME')
    process_greek(cm_grid, 'LAU1_NAME')

    cm_grid.to_csv(f'{base_folder}/{NUTS2}/GR_{NUTS2}_Timeline_GRID_{year_item}_filled_mean_rain_shp.csv', encoding=enc, index = False)


Year: 2009
No column named 'rainfall.1'


C:\Users\dimit\AppData\Local\Temp\ipykernel_8116\1240137723.py:26: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  greece_lau1['lau1_centroid'] = greece_lau1['lau1_geom'].centroid
c:\Users\dimit\Python Virtual Environments\noa_eywa\Lib\site-packages\geopandas\array.py:365: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


greece_nuts3 CRS: EPSG:4326
greece_lau1 CRS: EPSG:4326
greece_grid CRS: EPSG:4326
Year: 2010
No column named 'rainfall.1'


C:\Users\dimit\AppData\Local\Temp\ipykernel_8116\1240137723.py:26: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  greece_lau1['lau1_centroid'] = greece_lau1['lau1_geom'].centroid
c:\Users\dimit\Python Virtual Environments\noa_eywa\Lib\site-packages\geopandas\array.py:365: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


greece_nuts3 CRS: EPSG:4326
greece_lau1 CRS: EPSG:4326
greece_grid CRS: EPSG:4326
